# V10-E0 — Memory autopsy: where exactly are the ~943 MB?

V9 killed two VRAM levers (bf16 storage, recompute-`a`) and proved the scan
state (~67 MB) was never the problem. This notebook measures the peak before
any new lever is designed.

**Methodological rule for this autopsy:** report peak *live* memory by
tensor/category — bytes alive at the moment of peak — never cumulative
allocation. A large tensor freed before the peak contributes zero to it;
lifetimes decide the map, not sizes.

Analytical prediction (to confirm or kill):
FFN ≈ 232 MB | mixer projections ≈ 134 MB | norms/residuals ≈ 134 MB |
scan state ≈ 67 MB | remainder (workspace/transients/fragmentation) ≈ 440 MB.

**E0 gate:** named live-at-peak categories ≥ 85% of snapshot live total, and
accounted (live + fragmentation gap) within 10% of peak reserved. Any lever
whose target tensor isn't in the top-3 live-at-peak is falsified before it's
built. No GitHub push.


## 0. Setup

In [ ]:
import torch
import triton
print("torch", torch.__version__, "| triton", triton.__version__)
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
device = "cuda"
assert torch.cuda.is_available(), "V10-E0 needs the T4 GPU"

def set_seed(s):
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
set_seed(0)


## 1. Data — campaign corpus A only (vocab 104, same as V8-A Run 1)

In [ ]:
import os, urllib.request
def dl(url, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path):
        print("downloading", url, flush=True)
        urllib.request.urlretrieve(url, path)
    return path

URLS = {
    "tinyshakespeare.txt": "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "alice.txt":           "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "frankenstein.txt":    "https://www.gutenberg.org/cache/epub/84/pg84.txt",
    "pride.txt":           "https://www.gutenberg.org/cache/epub/1342/pg1342.txt",
}
parts = []
for name, url in URLS.items():
    p = dl(url, f"data/{name}")
    with open(p, encoding="utf-8", errors="ignore") as f:
        parts.append(f.read())
text_a = "\n".join(parts)
chars_a = sorted(set(text_a)); vocab_a = len(chars_a)
stoi_a = {c: i for i, c in enumerate(chars_a)}
data_a = torch.tensor([stoi_a[c] for c in text_a], dtype=torch.long)
nl_a = stoi_a.get("\n")
assert vocab_a == 104, f"vocab drift: {vocab_a}"

def make_get_batch(d, seq_len):
    def get_batch(split="train", batch_size=32):
        dd = d[:int(0.95 * len(d))] if split == "train" else d[int(0.95 * len(d)):]
        i = torch.randint(0, len(dd) - seq_len - 1, (batch_size,))
        x = torch.stack([dd[j:j+seq_len] for j in i]).to(device)
        y = torch.stack([dd[j+1:j+seq_len+1] for j in i]).to(device)
        return x, y
    return get_batch

get_batch = make_get_batch(data_a, 128)
print(f"vocab {vocab_a} | tokens {len(data_a)/1e6:.1f}M | nl_id={nl_a}")


## 2. Models + Triton kernels (inlined, no imports)

In [ ]:
"""
Linear-recurrent replacement for Transformer self-attention.

Experiment: take a standard decoder-only Transformer, rip out the
causal self-attention layer, and replace it with a *moving linear
equation* that compresses the past into a fixed-size running state:

    h_t = d * h_{t-1} + (1 - d) * (B x_t)        (linear recurrence)
    y_t = C h_t                                   (readout)

where d in (0,1)^S is a learned per-channel decay, B/C are learned
projections, and h has FIXED size S independent of sequence length.

This is O(T) time and O(1) memory w.r.t. context length, vs O(T^2)
for attention. The question: how much modeling power do we lose?
"""

import math
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------
# Token mixing layers
# ----------------------------------------------------------------------------

class SelectiveSegmentedState(nn.Module):
    """V5: gated segmented state-space mixer.

    Replaces the static decay with two dynamic mechanisms:

    1. Dynamic Selective Gate (the model's "will"):
         g_t = sigmoid(W_g x_t + b_g)      in (0,1), per token & channel
       acts as a learned per-token filter: low-information tokens get
       g_t -> 0 (forgotten fast), salient tokens get g_t -> 1 (kept).

    2. Hard Memory Reset Mask (segment boundaries):
         r_t = 1 if token t is a structural boundary (e.g. newline)
         h_t = (1 - r_t) * (g_t * h_{t-1} + (1 - g_t) * u_t)
       when r_t = 1 the state is multiplied by zero: the past memory
       buffer is completely destroyed and the new segment starts fresh.
       (The residual stream still carries x_t itself forward.)

    The recurrence stays affine in h (h_t = a_t*h_{t-1} + b_t with
    a_t = (1-r_t)*g_t, b_t = (1-r_t)*(1-g_t)*u_t), so the parallel
    associative scan still applies: O(T) work, log2(T) parallel passes.
    State dim stays frozen at `state_dim` (VRAM edge preserved).
    """
    def __init__(self, dim, state_dim=256, dropout=0.0):
        super().__init__()
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)  # bias = per-channel prior
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        # x: (B,T,dim); reset: (B,T) binary, 1 = boundary token
        u = self.in_proj(x)                    # (B,T,S)
        g = torch.sigmoid(self.gate_proj(x))   # (B,T,S) dynamic gate
        if reset is not None:
            keep = (1.0 - reset).unsqueeze(-1)  # (B,T,1)
            a = keep * g
            b = keep * (1.0 - g) * u
        else:
            a = g
            b = (1.0 - g) * u
        h_seq = parallel_scan_ab(a, b)          # (B,T,S)
        return self.dropout(self.out_proj(h_seq))


class CausalSelfAttention(nn.Module):
    """Standard multi-head causal self-attention (the baseline)."""
    def __init__(self, dim, n_heads, dropout=0.0):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.out = nn.Linear(dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)                       # (B,T,H,D)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))  # (B,H,T,D)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(y)


def parallel_scan_ab(A, B):
    """Parallel associative scan (Hillis-Steele) for h_t = A_t*h_{t-1} + B_t.

    A, B: (B, T, S) per-step coefficients. Returns H: (B, T, S).
    log2(T) fully-parallel passes instead of T sequential steps.
    """
    Bsz, T, S = A.shape
    A = A.contiguous()
    Bh = B.contiguous()
    idx = torch.arange(T, device=A.device).view(1, T, 1)
    o = 1
    while o < T:
        m = idx >= o
        A_sh = torch.roll(A, o, dims=1)
        B_sh = torch.roll(Bh, o, dims=1)
        # compose segment [t-o, t-1] then [t]:
        #   a_new = a_t * a_{t-o},  b_new = a_t * b_{t-o} + b_t
        A_new = A * A_sh
        B_new = A * B_sh + Bh
        A = torch.where(m, A_new, A)
        Bh = torch.where(m, B_new, Bh)
        o *= 2
    return Bh


def parallel_linear_recurrence(u, decay):
    """Parallel associative scan (Hillis-Steele) for h_t = d*h_{t-1} + c_t.

    Reformulates the sequential loop as a tree reduction:
      - each step is an affine map  h -> a*h + b, packed as pair (a, b)
      - pair composition is associative:
            (a1,b1) then (a2,b2)  =  (a2*a1, a2*b1 + b2)
      - a parallel prefix over pairs yields every h_t at once

    log2(T) fully-parallel passes instead of T sequential steps.
    u: (B, T, S) inputs already projected; decay: (S,) in (0, 1).
    Returns h: (B, T, S) — identical math to the for-loop version.
    """
    Bsz, T, S = u.shape
    c = (1.0 - decay) * u
    A = decay.expand(Bsz, T, S).contiguous()   # a-coefficients
    return parallel_scan_ab(A, c)


class LinearRunningState(nn.Module):
    """Attention replacement: fixed-size running state, linear recurrence.

    h_t = decay * h_{t-1} + (1 - decay) * (B x_t)
    y_t = C h_t

    - decay: learned per-state-channel forget factor in (0, 1)
    - B:     input  projection  dim -> state_dim
    - C:     output projection  state_dim -> dim
    State size is FIXED (does not grow with context length).
    mode="loop": native Python for-loop (sequential, slow on GPU)
    mode="scan": parallel associative scan (log2(T) parallel passes)
    """
    def __init__(self, dim, state_dim=None, dropout=0.0, mode="loop"):
        super().__init__()
        self.mode = mode
        self.state_dim = state_dim or dim
        # init decay ~ U(0.9, 0.999) -> long-ish memory at start
        init = torch.empty(self.state_dim).uniform_(math.log(0.9 / 0.1),
                                                   math.log(0.999 / 0.001))
        self.logit_decay = nn.Parameter(init)
        self.in_proj = nn.Linear(dim, self.state_dim, bias=False)
        self.out_proj = nn.Linear(self.state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        B, T, _ = x.shape
        decay = torch.sigmoid(self.logit_decay)          # (S,) in (0,1)
        u = self.in_proj(x)                              # (B,T,S)
        if self.mode == "scan":
            h_seq = parallel_linear_recurrence(u, decay)  # (B,T,S)
        else:
            keep = 1.0 - decay
            h = torch.zeros(B, self.state_dim, device=x.device, dtype=x.dtype)
            outs = []
            for t in range(T):
                h = decay * h + keep * u[:, t]
                outs.append(h)
            h_seq = torch.stack(outs, dim=1)              # (B,T,S)
        return self.dropout(self.out_proj(h_seq))


# ----------------------------------------------------------------------------
# Blocks & models (identical except for the mixing layer)
# ----------------------------------------------------------------------------

class Block(nn.Module):
    def __init__(self, dim, n_heads, mixer, dropout=0.0, ffn_hidden=None):
        super().__init__()
        ffn_hidden = ffn_hidden or 4 * dim
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.mix = mixer
        self.mlp = nn.Sequential(
            nn.Linear(dim, ffn_hidden), nn.GELU(),
            nn.Linear(ffn_hidden, dim), nn.Dropout(dropout))

    def forward(self, x, reset=None):
        x = x + self.mix(self.ln1(x), reset)
        x = x + self.mlp(self.ln2(x))
        return x


class TinyLM(nn.Module):
    """Decoder-only LM. mixer_fn(dim, n_heads) builds the token-mixing layer."""
    def __init__(self, vocab, dim=128, n_layers=4, n_heads=4,
                 seq_len=128, mixer_fn=None, dropout=0.0, ffn_hidden=None,
                 newline_id=None):
        super().__init__()
        self.seq_len = seq_len
        self.newline_id = newline_id  # boundary token id for reset mask
        self.tok_emb = nn.Embedding(vocab, dim)
        self.pos_emb = nn.Embedding(seq_len, dim)
        self.blocks = nn.ModuleList(
            [Block(dim, n_heads, mixer_fn(dim, n_heads), dropout, ffn_hidden)
             for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(
            torch.arange(T, device=idx.device))[None]
        reset = None
        if self.newline_id is not None:
            reset = (idx == self.newline_id).float()  # (B,T) boundary mask
        for blk in self.blocks:
            x = blk(x, reset)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new, temp=1.0, top_k=None):
        for _ in range(max_new):
            ctx = idx[:, -self.seq_len:]
            logits, _ = self(ctx)
            logits = logits[:, -1, :] / max(temp, 1e-6)
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx


def count_params(m):
    return sum(p.numel() for p in m.parameters())


def build_models(vocab, dim=128, n_layers=4, n_heads=4, seq_len=128,
                 state_dim=None, dropout=0.0):
    attn = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                  mixer_fn=lambda d, h: CausalSelfAttention(d, h, dropout),
                  dropout=dropout)
    lin = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                 mixer_fn=lambda d, h: LinearRunningState(d, state_dim, dropout),
                 dropout=dropout)
    return attn, lin


"""(inlined) V6/V7/V8-A: Triton-fused scans.

Fuses the V5 recurrence (per the V6 execution plan):
  Step A: the 256-wide hidden state lives in SRAM registers for the whole
          sequence -- loaded once per batch element, never spilled to HBM.
  Step B: sigmoid gate, hard reset mask, and the linear update
          h = keep*g*h + keep*(1-g)*u run in registers, every timestep.
  Step C: only h_t (plus a_t, saved for the backward pass) hit HBM.
          The parallel scan's O(log T) intermediate (B,T,S) tensors vanish.

Math is IDENTICAL to V5's SelectiveSegmentedState -- only the execution
path changes. Zero new parameters.
"""
import triton
import triton.language as tl

BLOCK_S = 256  # state dim; the whole state fits in SRAM


@triton.jit
def _sel_scan_fwd_kernel(u_ptr, gp_ptr, keep_ptr, h_ptr, a_ptr,
                         T, BLOCK: tl.constexpr):
    """Forward: h_t = a_t*h_{t-1} + b_t with a_t = keep_t*g_t,
    b_t = keep_t*(1-g_t)*u_t, g_t = sigmoid(gp_t)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    u_b = u_ptr + b * bs
    gp_b = gp_ptr + b * bs
    h_b = h_ptr + b * bs
    a_b = a_ptr + b * bs
    k_b = keep_ptr + b * T
    h = tl.zeros([BLOCK], dtype=tl.float32)          # Step A: state in SRAM
    for t in range(T):
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        gp_t = tl.load(gp_b + t * BLOCK + offs).to(tl.float32)
        k = tl.load(k_b + t).to(tl.float32)
        g = 1.0 / (1.0 + tl.exp(-gp_t))              # Step B: gate in registers
        at = k * g
        h = at * h + k * (1.0 - g) * u_t             # Step B: update in registers
        tl.store(h_b + t * BLOCK + offs, h)          # Step C: outputs to HBM
        tl.store(a_b + t * BLOCK + offs, at)


@triton.jit
def _sel_scan_bwd_kernel(dh_ptr, hs_ptr, ap_ptr, u_ptr, gp_ptr, keep_ptr,
                         du_ptr, dgp_ptr, T, BLOCK: tl.constexpr):
    """Backward. hs is (B,T+1,S) with hs[t] = h_{t-1} (hs[0] = 0);
    ap is (B,T+1,S) with ap[t+1] = a_{t+1} (ap[T] = 0).
    D_t = gout_t + a_{t+1}*D_{t+1}; du_t = k(1-g)D_t;
    dgp_t = k*g*(1-g)*(h_{t-1}-u_t)*D_t."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    bsp = (T + 1) * BLOCK
    dh_b = dh_ptr + b * bs
    hs_b = hs_ptr + b * bsp
    ap_b = ap_ptr + b * bsp
    u_b = u_ptr + b * bs
    gp_b = gp_ptr + b * bs
    k_b = keep_ptr + b * T
    du_b = du_ptr + b * bs
    dgp_b = dgp_ptr + b * bs
    dnext = tl.zeros([BLOCK], dtype=tl.float32)      # D_{t+1}, SRAM-resident
    for ti in range(T):
        t = T - 1 - ti
        gout = tl.load(dh_b + t * BLOCK + offs).to(tl.float32)
        a_next = tl.load(ap_b + (t + 1) * BLOCK + offs).to(tl.float32)
        D = gout + a_next * dnext
        h_prev = tl.load(hs_b + t * BLOCK + offs).to(tl.float32)
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        gp_t = tl.load(gp_b + t * BLOCK + offs).to(tl.float32)
        k = tl.load(k_b + t).to(tl.float32)
        g = 1.0 / (1.0 + tl.exp(-gp_t))
        omg = 1.0 - g
        tl.store(du_b + t * BLOCK + offs, k * omg * D)
        tl.store(dgp_b + t * BLOCK + offs, k * g * omg * (h_prev - u_t) * D)
        dnext = D


class _SelScanFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, u, gp, keep):
        assert u.is_cuda and u.dtype == torch.float32, "V6 kernel: CUDA fp32"
        assert u.shape == gp.shape and u.dim() == 3
        assert keep.shape == u.shape[:2]
        assert u.shape[2] == BLOCK_S
        B, T, S = u.shape
        u, gp, keep = u.contiguous(), gp.contiguous(), keep.contiguous()
        h = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        a = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        _sel_scan_fwd_kernel[(B,)](u, gp, keep, h, a, T,
                                   BLOCK=BLOCK_S, num_warps=4)
        ctx.save_for_backward(u, gp, keep, h, a)
        return h

    @staticmethod
    def backward(ctx, dh):
        u, gp, keep, h, a = ctx.saved_tensors
        B, T, S = u.shape
        dh = dh.contiguous()
        hs = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        hs[:, 1:] = h    # hs[t] = h_{t-1}
        ap = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        ap[:, :T] = a    # ap[t+1] = a_{t+1}
        du = torch.empty_like(u)
        dgp = torch.empty_like(gp)
        _sel_scan_bwd_kernel[(B,)](dh, hs, ap, u, gp, keep, du, dgp, T,
                                   BLOCK=BLOCK_S, num_warps=4)
        return du, dgp, None


def triton_selective_scan(u, gp, keep):
    """Fused selective scan. u, gp: (B,T,256) fp32 CUDA; keep: (B,T) fp32.
    Returns h: (B,T,256). Differentiable."""
    return _SelScanFn.apply(u, gp, keep)


class SelectiveSegmentedStateTriton(nn.Module):
    """V6 mixer: IDENTICAL math and parameters to V5's SelectiveSegmentedState.

    The only change is execution: the recurrence runs in the fused Triton
    kernel (Step A/B/C) instead of the PyTorch Hillis-Steele scan.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V6 kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        gp = self.gate_proj(x).float().contiguous()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_selective_scan(u, gp, keep)
        return self.dropout(self.out_proj(h_seq.to(x.dtype)))


class SelectiveSegmentedStateV7(nn.Module):
    """V7 mixer: V6's Triton-fused selective scan + two literature grafts.

    Graft #1 — output gate + state normalization (HGRN2 / Mamba-2 / GDN):
        y_t = C · (o_t ⊙ RMSNorm(h_t)),   o_t = σ(W_o x_t + b_o)
    States accumulate over segments so readout scale drifts; every strong
    post-Mamba block gates the readout. RMSNorm here is parameter-free
    (no scale vector) to keep the parameter budget exact.

    Graft #3 — layer-graded forget floor (HGRN, NeurIPS'23):
        g_t = σ(W_g x_t + b_g + floor_l)
    floor_l is a per-layer learnable scalar, initialized monotonically
    increasing bottom→top, so low layers forget fast (spelling-level) and
    top layers retain (phrase-level). Complements the newline hard reset.

    The recurrence math is UNCHANGED → the same fused Triton kernel as V6.
    With FFN 1023 the budget lands EXACTLY on the attention baseline.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0, forget_floor_init=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V7 kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)
        self.out_gate_proj = nn.Linear(dim, state_dim)
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.forget_floor = nn.Parameter(
            torch.tensor(float(forget_floor_init)))
        self.dropout = nn.Dropout(dropout)
        self.eps = 1e-6

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        gp = (self.gate_proj(x) + self.forget_floor).float().contiguous()
        og = self.out_gate_proj(x).float()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_selective_scan(u, gp, keep)          # (B,T,S)
        h_norm = h_seq / torch.sqrt(
            (h_seq ** 2).mean(dim=-1, keepdim=True) + self.eps)
        y = self.out_proj((torch.sigmoid(og) * h_norm).to(x.dtype))
        return self.dropout(y)


# ----------------------------------------------------------------------------
# V8-A: diagonal gated delta-rule scan
# ----------------------------------------------------------------------------

@triton.jit
def _delta_scan_fwd_kernel(u_ptr, ap_ptr, bp_ptr, kp_ptr, keep_ptr,
                           h_ptr, a_ptr, T, BLOCK: tl.constexpr):
    """Forward: h_t = a_t*h_{t-1} + b_t with
        alpha = sigmoid(ap_t), beta = sigmoid(bp_t), kk = sigmoid(kp_t),
        v = silu(u_t),
        a_t = keep_t * alpha * (1 - beta*kk^2),
        b_t = keep_t * beta * kk * v.
    Diagonal restriction of Gated DeltaNet's S_t = a_t S_{t-1}(I-b_t k k^T)
    + b_t v k^T to per-channel 1x1 systems. Strictly contractive:
    sigmoid bounds keep (1-beta*kk^2) in (0,1)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    u_b = u_ptr + b * bs
    ap_b = ap_ptr + b * bs
    bp_b = bp_ptr + b * bs
    kp_b = kp_ptr + b * bs
    h_b = h_ptr + b * bs
    a_b = a_ptr + b * bs
    k_b = keep_ptr + b * T
    h = tl.zeros([BLOCK], dtype=tl.float32)          # state in SRAM
    for t in range(T):
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        ap_t = tl.load(ap_b + t * BLOCK + offs).to(tl.float32)
        bp_t = tl.load(bp_b + t * BLOCK + offs).to(tl.float32)
        kp_t = tl.load(kp_b + t * BLOCK + offs).to(tl.float32)
        kr = tl.load(k_b + t).to(tl.float32)
        alpha = 1.0 / (1.0 + tl.exp(-ap_t))
        beta = 1.0 / (1.0 + tl.exp(-bp_t))
        kk = 1.0 / (1.0 + tl.exp(-kp_t))
        su = 1.0 / (1.0 + tl.exp(-u_t))
        v = u_t * su                                 # SiLU in registers
        at = kr * alpha * (1.0 - beta * kk * kk)
        bt = kr * beta * kk * v
        h = at * h + bt
        tl.store(h_b + t * BLOCK + offs, h)
        tl.store(a_b + t * BLOCK + offs, at)


@triton.jit
def _delta_scan_bwd_kernel(dh_ptr, hs_ptr, apn_ptr, u_ptr, ap_ptr, bp_ptr,
                           kp_ptr, keep_ptr, du_ptr, dap_ptr, dbp_ptr,
                           dkp_ptr, T, BLOCK: tl.constexpr):
    """Backward. hs is (B,T+1,S) with hs[t] = h_{t-1} (hs[0] = 0);
    apn is (B,T+1,S) with apn[t+1] = a_{t+1} (apn[T] = 0).
    Adjoint: D_t = gout_t + a_{t+1}*D_{t+1}; then per-step
      du  = D*kr*beta*kk*silu'(u)
      dap = D*h_prev*kr*alpha*(1-alpha)*(1-beta*kk^2)
      dbp = D*kr*beta*(1-beta)*(kk*v - alpha*kk^2*h_prev)
      dkp = D*kr*beta*kk*(1-kk)*(v - 2*alpha*kk*h_prev)
    (hand-derived; verified against autograd on CPU before transcription)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    bsp = (T + 1) * BLOCK
    dh_b = dh_ptr + b * bs
    hs_b = hs_ptr + b * bsp
    apn_b = apn_ptr + b * bsp
    u_b = u_ptr + b * bs
    ap_b = ap_ptr + b * bs
    bp_b = bp_ptr + b * bs
    kp_b = kp_ptr + b * bs
    k_b = keep_ptr + b * T
    du_b = du_ptr + b * bs
    dap_b = dap_ptr + b * bs
    dbp_b = dbp_ptr + b * bs
    dkp_b = dkp_ptr + b * bs
    dnext = tl.zeros([BLOCK], dtype=tl.float32)      # D_{t+1}, SRAM-resident
    for ti in range(T):
        t = T - 1 - ti
        gout = tl.load(dh_b + t * BLOCK + offs).to(tl.float32)
        a_next = tl.load(apn_b + (t + 1) * BLOCK + offs).to(tl.float32)
        D = gout + a_next * dnext
        h_prev = tl.load(hs_b + t * BLOCK + offs).to(tl.float32)
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        ap_t = tl.load(ap_b + t * BLOCK + offs).to(tl.float32)
        bp_t = tl.load(bp_b + t * BLOCK + offs).to(tl.float32)
        kp_t = tl.load(kp_b + t * BLOCK + offs).to(tl.float32)
        kr = tl.load(k_b + t).to(tl.float32)
        alpha = 1.0 / (1.0 + tl.exp(-ap_t))
        beta = 1.0 / (1.0 + tl.exp(-bp_t))
        kk = 1.0 / (1.0 + tl.exp(-kp_t))
        su = 1.0 / (1.0 + tl.exp(-u_t))
        v = u_t * su
        silu_p = su * (1.0 + u_t * (1.0 - su))
        k2 = kk * kk
        omb = 1.0 - beta * k2
        tl.store(du_b + t * BLOCK + offs,
                 D * kr * beta * kk * silu_p)
        tl.store(dap_b + t * BLOCK + offs,
                 D * h_prev * kr * alpha * (1.0 - alpha) * omb)
        tl.store(dbp_b + t * BLOCK + offs,
                 D * kr * beta * (1.0 - beta) * (kk * v - alpha * k2 * h_prev))
        tl.store(dkp_b + t * BLOCK + offs,
                 D * kr * beta * kk * (1.0 - kk) * (v - 2.0 * alpha * kk * h_prev))
        dnext = D


def _delta_scan_cpu(u, ap, bp, kp, keep):
    """CPU reference for the delta scan: differentiable pure-PyTorch loop.

    Used ONLY for local (CPU) verification. On CUDA the fused Triton
    kernel above is used instead (gate-checked against this math on Colab).
    """
    B, T, S = u.shape
    kr = keep.unsqueeze(-1)
    alpha = torch.sigmoid(ap)
    beta = torch.sigmoid(bp)
    kk = torch.sigmoid(kp)
    v = u * torch.sigmoid(u)                          # SiLU
    h = torch.zeros(B, S, dtype=u.dtype, device=u.device)
    hs = []
    for t in range(T):
        a = kr[:, t] * alpha[:, t] * (1.0 - beta[:, t] * kk[:, t] ** 2)
        bb = kr[:, t] * beta[:, t] * kk[:, t] * v[:, t]
        h = a * h + bb
        hs.append(h)
    return torch.stack(hs, dim=1)


class _DeltaScanFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, u, ap, bp, kp, keep):
        assert u.is_cuda and u.dtype == torch.float32, \
            "V8-A kernel: CUDA fp32"
        for t in (u, ap, bp, kp):
            assert t.shape == u.shape and t.dim() == 3
        assert keep.shape == u.shape[:2]
        assert u.shape[2] == BLOCK_S
        B, T, S = u.shape
        u, ap, bp, kp = u.contiguous(), ap.contiguous(), bp.contiguous(), kp.contiguous()
        keep = keep.contiguous()
        h = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        a = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        _delta_scan_fwd_kernel[(B,)](u, ap, bp, kp, keep, h, a, T,
                                    BLOCK=BLOCK_S, num_warps=4)
        ctx.save_for_backward(u, ap, bp, kp, keep, h, a)
        return h

    @staticmethod
    def backward(ctx, dh):
        u, ap, bp, kp, keep, h, a = ctx.saved_tensors
        B, T, S = u.shape
        dh = dh.contiguous()
        hs = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        hs[:, 1:] = h      # hs[t] = h_{t-1}
        apn = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        apn[:, :T] = a     # apn[t+1] = a_{t+1}
        du = torch.empty_like(u)
        dap = torch.empty_like(ap)
        dbp = torch.empty_like(bp)
        dkp = torch.empty_like(kp)
        _delta_scan_bwd_kernel[(B,)](dh, hs, apn, u, ap, bp, kp, keep,
                                    du, dap, dbp, dkp, T,
                                    BLOCK=BLOCK_S, num_warps=4)
        return du, dap, dbp, dkp, None


def triton_delta_scan(u, ap, bp, kp, keep):
    """Fused diagonal delta-rule scan.

    u: raw Bx (SiLU applied in-kernel); ap/bp/kp: pre-sigmoid
    alpha/beta/key projections; keep: (B,T) reset mask.
    (B,T,256) fp32; CUDA -> Triton kernel, CPU -> differentiable
    reference loop (verification only). Returns h: (B,T,256).
    """
    if u.is_cuda:
        return _DeltaScanFn.apply(u, ap, bp, kp, keep)
    return _delta_scan_cpu(u, ap, bp, kp, keep)


class SelectiveSegmentedStateV8A(nn.Module):
    """V8-A mixer: diagonal gated delta rule on V7's chassis.

    Per channel (delta rule = erase-before-write, Gated DeltaNet Eq.10
    restricted to 1x1 systems):
        alpha_t = sigmoid(W_a x_t + b_a + floor_l)   # reuse gate_proj+floor
        beta_t  = sigmoid(W_b x_t + b_b),  b_b = 0   # NEW write_proj
        k_t     = sigmoid(W_k x_t + b_k),  b_k = +2  # NEW key_proj
        v_t     = SiLU(B x_t)                        # reuse in_proj + SiLU
        a_t = alpha_t * (1 - beta_t * k_t^2)
        b_t = beta_t * k_t * v_t
        h_t = (1 - r_t) * (a_t .* h_{t-1} + b_t)
        y_t = C . (o_t .* RMSNorm(h_t)), o_t = sigmoid(W_o x_t)

    Init puts the model near V7's convex-blend regime (k~=0.88, beta~=0.5)
    and the sigmoid bounds keep every step strictly contractive.
    With FFN 766 the budget lands within the relaxed <=3000 gate.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0, forget_floor_init=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V8-A kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)   # B
        self.gate_proj = nn.Linear(dim, state_dim)             # W_alpha
        self.write_proj = nn.Linear(dim, state_dim)            # W_beta (NEW)
        self.key_proj = nn.Linear(dim, state_dim)              # W_k (NEW)
        self.out_gate_proj = nn.Linear(dim, state_dim)         # W_o
        self.out_proj = nn.Linear(state_dim, dim, bias=False)  # C
        self.forget_floor = nn.Parameter(
            torch.tensor(float(forget_floor_init)))
        self.write_proj.bias.data.zero_()       # b_beta = 0  -> beta ~= 0.5
        self.key_proj.bias.data.fill_(2.0)       # b_k = +2.0  -> k ~= 0.88
        self.dropout = nn.Dropout(dropout)
        self.eps = 1e-6
        self.last_state = None  # stashed h_seq for stability logging (no grad)

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        ap = (self.gate_proj(x) + self.forget_floor).float().contiguous()
        bp = self.write_proj(x).float().contiguous()
        kp = self.key_proj(x).float().contiguous()
        og = self.out_gate_proj(x).float()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_delta_scan(u, ap, bp, kp, keep)      # (B,T,S)
        self.last_state = h_seq.detach()  # stability logging only; no numerics change
        h_norm = h_seq / torch.sqrt(
            (h_seq ** 2).mean(dim=-1, keepdim=True) + self.eps)
        y = self.out_proj((torch.sigmoid(og) * h_norm).to(x.dtype))
        return self.dropout(y)


## 2b. Line->category map (baked at generation from the exact source above)

In [ ]:
_LINEMAP = {52: 'mixer-proj', 53: 'mixer-proj', 54: 'mixer-proj', 59: 'mixer-proj', 60: 'mixer-proj', 69: 'mixer-proj', 72: 'CausalSelfAttention-misc', 73: 'CausalSelfAttention-misc', 74: 'CausalSelfAttention-misc', 75: 'CausalSelfAttention-misc', 76: 'CausalSelfAttention-misc', 77: 'CausalSelfAttention-misc', 78: 'logits/loss', 79: 'CausalSelfAttention-misc', 80: 'CausalSelfAttention-misc', 81: 'CausalSelfAttention-misc', 82: 'CausalSelfAttention-misc', 83: 'CausalSelfAttention-misc', 84: 'CausalSelfAttention-misc', 85: 'logits/loss', 86: 'CausalSelfAttention-misc', 87: 'CausalSelfAttention-misc', 88: 'CausalSelfAttention-misc', 89: 'CausalSelfAttention-misc', 90: 'CausalSelfAttention-misc', 91: 'CausalSelfAttention-misc', 138: 'LinearRunningState-misc', 139: 'LinearRunningState-misc', 140: 'LinearRunningState-misc', 141: 'LinearRunningState-misc', 142: 'LinearRunningState-misc', 143: 'LinearRunningState-misc', 144: 'LinearRunningState-misc', 145: 'LinearRunningState-misc', 146: 'LinearRunningState-misc', 147: 'LinearRunningState-misc', 148: 'LinearRunningState-misc', 149: 'LinearRunningState-misc', 150: 'LinearRunningState-misc', 151: 'LinearRunningState-misc', 152: 'LinearRunningState-misc', 153: 'LinearRunningState-misc', 154: 'LinearRunningState-misc', 155: 'LinearRunningState-misc', 156: 'LinearRunningState-misc', 157: 'LinearRunningState-misc', 158: 'LinearRunningState-misc', 159: 'mixer-proj', 160: 'mixer-proj', 161: 'LinearRunningState-misc', 162: 'LinearRunningState-misc', 163: 'LinearRunningState-misc', 164: 'LinearRunningState-misc', 165: 'LinearRunningState-misc', 166: 'mixer-proj', 167: 'LinearRunningState-misc', 168: 'LinearRunningState-misc', 169: 'LinearRunningState-misc', 170: 'LinearRunningState-misc', 171: 'LinearRunningState-misc', 172: 'LinearRunningState-misc', 173: 'LinearRunningState-misc', 174: 'LinearRunningState-misc', 175: 'LinearRunningState-misc', 176: 'LinearRunningState-misc', 177: 'mixer-proj', 184: 'Block-misc', 185: 'Block-misc', 186: 'Block-misc', 187: 'Block-misc', 188: 'norms', 189: 'norms', 190: 'Block-misc', 191: 'ffn', 192: 'ffn', 193: 'Block-misc', 194: 'Block-misc', 195: 'Block-misc', 196: 'norms', 197: 'ffn', 198: 'Block-misc', 201: 'TinyLM-misc', 202: 'TinyLM-misc', 203: 'TinyLM-misc', 204: 'TinyLM-misc', 205: 'TinyLM-misc', 206: 'TinyLM-misc', 207: 'TinyLM-misc', 208: 'TinyLM-misc', 209: 'embeddings', 210: 'embeddings', 211: 'TinyLM-misc', 212: 'TinyLM-misc', 213: 'TinyLM-misc', 214: 'norms', 215: 'logits/loss', 216: 'embeddings', 217: 'TinyLM-misc', 218: 'TinyLM-misc', 219: 'TinyLM-misc', 220: 'TinyLM-misc', 221: 'embeddings', 222: 'TinyLM-misc', 223: 'TinyLM-misc', 224: 'TinyLM-misc', 225: 'TinyLM-misc', 226: 'embeddings', 227: 'TinyLM-misc', 228: 'TinyLM-misc', 229: 'TinyLM-misc', 230: 'TinyLM-misc', 231: 'TinyLM-misc', 232: 'TinyLM-misc', 233: 'norms', 234: 'TinyLM-misc', 235: 'TinyLM-misc', 236: 'logits/loss', 237: 'TinyLM-misc', 238: 'TinyLM-misc', 239: 'TinyLM-misc', 240: 'TinyLM-misc', 241: 'TinyLM-misc', 242: 'TinyLM-misc', 243: 'TinyLM-misc', 244: 'TinyLM-misc', 245: 'TinyLM-misc', 246: 'TinyLM-misc', 247: 'TinyLM-misc', 248: 'TinyLM-misc', 249: 'TinyLM-misc', 250: 'TinyLM-misc', 251: 'TinyLM-misc', 349: 'scan-state', 350: 'scan-state', 351: 'scan-state', 352: 'scan-state', 353: 'scan-state', 354: 'scan-state', 355: 'scan-state', 356: 'scan-state', 357: 'scan-state', 358: 'scan-state', 359: 'scan-state', 360: 'scan-state', 361: 'scan-state', 362: 'scan-state', 363: 'scan-state', 364: 'scan-state', 365: 'scan-state', 366: 'scan-state', 367: 'scan-state', 368: 'scan-state', 369: 'scan-state', 370: 'scan-state', 371: 'scan-state', 372: 'scan-state', 373: 'scan-state', 374: 'scan-state', 375: 'scan-state', 376: 'scan-state', 377: 'scan-state', 378: 'scan-state', 397: 'mixer-proj', 398: 'mixer-proj', 399: 'mixer-proj', 403: 'mixer-proj', 404: 'mixer-proj', 411: 'mixer-proj', 418: 'norms', 420: 'norms', 436: 'mixer-proj', 437: 'mixer-proj', 438: 'mixer-proj', 439: 'mixer-proj', 446: 'mixer-proj', 447: 'mixer-proj', 448: 'mixer-proj', 455: 'norms', 457: 'mixer-proj', 586: 'scan-state', 587: 'scan-state', 588: 'scan-state', 589: 'scan-state', 590: 'scan-state', 591: 'scan-state', 592: 'scan-state', 593: 'scan-state', 594: 'scan-state', 595: 'scan-state', 596: 'scan-state', 597: 'scan-state', 598: 'scan-state', 599: 'scan-state', 600: 'scan-state', 601: 'scan-state', 602: 'scan-state', 603: 'scan-state', 604: 'scan-state', 605: 'scan-state', 606: 'scan-state', 607: 'scan-state', 608: 'scan-state', 609: 'scan-state', 610: 'scan-state', 611: 'scan-state', 612: 'scan-state', 613: 'scan-state', 614: 'scan-state', 615: 'scan-state', 616: 'scan-state', 617: 'scan-state', 618: 'scan-state', 619: 'scan-state', 620: 'scan-state', 621: 'scan-state', 624: 'scan-state', 637: 'SelectiveSegmentedStateV8A-misc', 638: 'SelectiveSegmentedStateV8A-misc', 639: 'SelectiveSegmentedStateV8A-misc', 640: 'SelectiveSegmentedStateV8A-misc', 641: 'SelectiveSegmentedStateV8A-misc', 642: 'mixer-proj', 643: 'mixer-proj', 644: 'mixer-proj', 645: 'mixer-proj', 646: 'SelectiveSegmentedStateV8A-misc', 647: 'SelectiveSegmentedStateV8A-misc', 648: 'SelectiveSegmentedStateV8A-misc', 649: 'norms', 650: 'SelectiveSegmentedStateV8A-misc', 651: 'SelectiveSegmentedStateV8A-misc', 652: 'SelectiveSegmentedStateV8A-misc', 653: 'SelectiveSegmentedStateV8A-misc', 654: 'SelectiveSegmentedStateV8A-misc', 655: 'SelectiveSegmentedStateV8A-misc', 656: 'SelectiveSegmentedStateV8A-misc', 657: 'SelectiveSegmentedStateV8A-misc', 658: 'SelectiveSegmentedStateV8A-misc', 659: 'mixer-proj', 660: 'mixer-proj', 661: 'mixer-proj', 662: 'mixer-proj', 663: 'mixer-proj', 664: 'mixer-proj', 665: 'SelectiveSegmentedStateV8A-misc', 666: 'SelectiveSegmentedStateV8A-misc', 667: 'mixer-proj', 668: 'mixer-proj', 669: 'SelectiveSegmentedStateV8A-misc', 670: 'SelectiveSegmentedStateV8A-misc', 671: 'SelectiveSegmentedStateV8A-misc', 672: 'SelectiveSegmentedStateV8A-misc', 673: 'SelectiveSegmentedStateV8A-misc', 674: 'mixer-proj', 675: 'mixer-proj', 676: 'mixer-proj', 677: 'mixer-proj', 678: 'mixer-proj', 679: 'SelectiveSegmentedStateV8A-misc', 680: 'SelectiveSegmentedStateV8A-misc', 681: 'SelectiveSegmentedStateV8A-misc', 682: 'SelectiveSegmentedStateV8A-misc', 683: 'SelectiveSegmentedStateV8A-misc', 684: 'scan-state', 685: 'SelectiveSegmentedStateV8A-misc', 686: 'norms', 687: 'SelectiveSegmentedStateV8A-misc', 688: 'mixer-proj', 689: 'SelectiveSegmentedStateV8A-misc'}
print(f"LINEMAP loaded: {len(_LINEMAP)} source lines categorized")
from collections import Counter
print(Counter(_LINEMAP.values()))


## 3. Build V8-A (identical to Run 1) + train_step

In [ ]:
DIM, LAYERS, HEADS, SEQ, FFN_H = 256, 8, 8, 128, 766
v8a = TinyLM(vocab_a, DIM, LAYERS, HEADS, SEQ,
             mixer_fn=lambda d, h: SelectiveSegmentedStateV8A(d, state_dim=256),
             ffn_hidden=FFN_H, newline_id=nl_a).to(device)
import math as _math
_floors = torch.linspace(_math.log(0.3/0.7), _math.log(0.9/0.1), LAYERS)
for _i, _blk in enumerate(v8a.blocks):
    _blk.mix.forget_floor.data.fill_(_floors[_i])
n_params = sum(p.numel() for p in v8a.parameters())
print(f"params: {n_params} (expect ~6.37M)")

opt = torch.optim.AdamW(v8a.parameters(), lr=3e-4)

def train_step(bs=32):
    opt.zero_grad(set_to_none=True)
    xb, yb = get_batch("train", batch_size=bs)
    _, loss = v8a(xb, yb)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(v8a.parameters(), 1.0)
    opt.step()
    return float(loss)

train_step()  # warmup: Triton compile + allocator warm
print("warmup done")


## E0.1 — Coarse peak split (allocated vs reserved)

In [ ]:
torch.cuda.reset_peak_memory_stats()
for _ in range(3):
    train_step()
peak_alloc = torch.cuda.max_memory_allocated() / 1e6
peak_res = torch.cuda.max_memory_reserved() / 1e6
frag_gap = peak_res - peak_alloc
print(f"peak alloc    {peak_alloc:.0f} MB")
print(f"peak reserved {peak_res:.0f} MB")
print(f"fragmentation gap (reserved-alloc): {frag_gap:.0f} MB  <- first free target (E2)")


## E0.2 — Allocator counters

In [ ]:
s = torch.cuda.memory_stats()
for k in ("allocated_bytes.all.peak", "reserved_bytes.all.peak",
          "inactive_split_bytes.all.peak", "num_alloc_retries", "num_ooms"):
    v = s[k]
    print(f"{k:32s}: {v/1e6:.1f} MB" if "bytes" in k else f"{k:32s}: {v}")


## E0.3 — Peak-LIVE attribution by category (the centerpiece)

Snapshots dumped at three points of one step (post-forward, post-backward,
post-optimizer). Live blocks at each dump are attributed via Python stack ->
source line -> category. The max-live dump is the peak composition.


In [ ]:
import pickle

def snapshot_live(tag):
    """Dump a memory snapshot; return {category: live_bytes} at dump time."""
    path = f"/content/snap_{tag}.pickle"
    torch.cuda.memory._dump_snapshot(path)
    snap = pickle.load(open(path, "rb"))
    cats, total = {}, 0
    n_blocks = 0
    _dbg = {"n_frames_total": 0, "blocks_with_user_frame": 0}
    for seg in snap["segments"]:
        for b in seg.get("blocks", []):
            if b.get("state") != "active_allocated":
                continue
            n_blocks += 1
            sz = b["size"]
            total += sz
            cat = "other"
            frames = b.get("frames", []) or []
            _dbg["n_frames_total"] += len(frames)
            hit_user = False
            # torch orders frames outermost->innermost; scan innermost-first so a
            # block attributes to the module line that created it, not the
            # training-loop line that called the model.
            for fr in reversed(frames):
                if isinstance(fr, (list, tuple)) and len(fr) >= 2:
                    fn, ln = str(fr[0]), fr[1]
                elif isinstance(fr, dict):
                    fn, ln = str(fr.get("filename", "")), fr.get("line", 0)
                else:
                    continue
                if "torch/optim" in fn or "adamw" in fn.lower():
                    cat = "optimizer"
                    break
                if "ipython-input" in fn:
                    hit_user = True
                    try:
                        c = _LINEMAP.get(int(ln), "other")
                    except (TypeError, ValueError):
                        c = "other"
                    if c != "other":
                        cat = c
                        break
                    # line categorized as other: keep scanning outward for a named frame
            if hit_user:
                _dbg["blocks_with_user_frame"] += 1
            cats[cat] = cats.get(cat, 0) + sz
    print(f"[debug] frames seen: {_dbg['n_frames_total']}, "
          f"blocks with >=1 user frame: {_dbg['blocks_with_user_frame']}/{n_blocks}")
    return cats, total, n_blocks

torch.cuda.memory._record_memory_history(enabled=True)
opt.zero_grad(set_to_none=True)
xb, yb = get_batch("train")
_, loss = v8a(xb, yb)
cats_fwd, tot_fwd, nb_fwd = snapshot_live("fwd")
loss.backward()
cats_bwd, tot_bwd, nb_bwd = snapshot_live("bwd")
torch.nn.utils.clip_grad_norm_(v8a.parameters(), 1.0)
opt.step()
cats_opt, tot_opt, nb_opt = snapshot_live("opt")
torch.cuda.memory._record_memory_history(enabled=None)

_snaps = {"post-forward": (cats_fwd, tot_fwd, nb_fwd),
          "post-backward": (cats_bwd, tot_bwd, nb_bwd),
          "post-optimizer": (cats_opt, tot_opt, nb_opt)}
for tag, (cats, tot, nb) in _snaps.items():
    print(f"--- live at {tag}: {tot/1e6:.0f} MB across {nb} blocks ---")
    for c, v in sorted(cats.items(), key=lambda kv: -kv[1]):
        print(f"  {c:28s} {v/1e6:7.1f} MB ({100*v/tot:4.1f}%)")
    print()


## E0.4 — Per-op attribution (profiler, complement to E0.3)

In [ ]:
from torch.profiler import profile, ProfilerActivity
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
             profile_memory=True, record_shapes=True, with_stack=True) as prof:
    for _ in range(2):
        train_step()
print(prof.key_averages().table(sort_by="self_cuda_memory_usage", row_limit=12))


## E0.5 — Owner-by-elimination (live-at-peak deltas)

Peak with the FFN / mixer ablated. The delta vs full *is* that component's
live-at-peak contribution, independent of the snapshot attribution.


In [ ]:
def peak_of(make_fn, steps=2):
    m = make_fn().to(device)
    o = torch.optim.AdamW(m.parameters(), lr=3e-4)
    m(get_batch("train", batch_size=32)[0])  # warmup fwd (Triton compile)
    torch.cuda.reset_peak_memory_stats()
    for _ in range(steps):
        o.zero_grad(set_to_none=True)
        _, loss = m(*get_batch("train", batch_size=32))
        loss.backward()
        o.step()
    p = torch.cuda.max_memory_allocated() / 1e6
    del m, o
    import gc; gc.collect()
    return p

def build_variant(mode):
    m = TinyLM(vocab_a, DIM, LAYERS, HEADS, SEQ,
               mixer_fn=lambda d, h: SelectiveSegmentedStateV8A(d, state_dim=256),
               ffn_hidden=FFN_H, newline_id=nl_a)
    if mode == "noffn":
        for blk in m.blocks:
            blk.mlp = torch.nn.Identity()
    if mode == "nomix":
        class Pass(torch.nn.Module):
            def forward(self, x, reset=None):
                return torch.zeros_like(x)
        for blk in m.blocks:
            blk.mix = Pass()
    return m

p_full = peak_of(lambda: build_variant("full"))
p_noffn = peak_of(lambda: build_variant("noffn"))
p_nomix = peak_of(lambda: build_variant("nomix"))
print(f"full     {p_full:.0f} MB")
print(f"no-FFN   {p_noffn:.0f} MB  (FFN live-at-peak ~ {p_full-p_noffn:.0f} MB)")
print(f"no-mixer {p_nomix:.0f} MB  (mixer live-at-peak ~ {p_full-p_nomix:.0f} MB)")


## E0.6 — Batch-dim ablation (isolate batch-linear activations)

In [ ]:
res = {}
for bs in (32, 16, 8, 4):
    train_step(bs)  # warm shapes
    torch.cuda.reset_peak_memory_stats()
    for _ in range(2):
        train_step(bs)
    res[bs] = torch.cuda.max_memory_allocated() / 1e6
    print(f"batch {bs:2d}: peak {res[bs]:.0f} MB", flush=True)
print("batch-linear slope ~", f"{(res[32]-res[4])/28:.1f} MB per sample")


## E0.7 — VERDICT: the optimization map

Gate: named live-at-peak categories >= 85% of snapshot live total, and
accounted (live + fragmentation gap) within 10% of peak reserved.


In [ ]:
# Recompute the coarse peak for the gate (fresh stats, same protocol)
torch.cuda.reset_peak_memory_stats()
for _ in range(3):
    train_step()
g_alloc = torch.cuda.max_memory_allocated() / 1e6
g_res = torch.cuda.max_memory_reserved() / 1e6
g_frag = g_res - g_alloc

best_tag, (best_cats, best_tot, best_nb) = max(
    _snaps.items(), key=lambda kv: kv[1][1])
named = sum(v for k, v in best_cats.items() if k != "other")
other = best_cats.get("other", 0)
accounted = best_tot / 1e6 + g_frag

print(f"peak composition from: {best_tag} ({best_tot/1e6:.0f} MB live)")
print(f"peak reserved (coarse): {g_res:.0f} MB | frag gap: {g_frag:.0f} MB")
print(f"named categories: {named/1e6:.0f} MB ({100*named/best_tot:.1f}% of live)")
print(f"uncategorized 'other': {other/1e6:.0f} MB ({100*other/best_tot:.1f}% of live)")
print(f"accounted (live+frag) vs reserved: {accounted:.0f} vs {g_res:.0f} MB")
print()
print("=== ranked live-at-peak map ===")
ranked = sorted(best_cats.items(), key=lambda kv: -kv[1])
for i, (c, v) in enumerate(ranked, 1):
    print(f"  {i}. {c:28s} {v/1e6:7.1f} MB")
print()
print("=== analytical prediction (confirm or kill) ===")
for c, v in [("ffn", 232), ("mixer-proj", 134), ("norms", 134),
             ("scan-state", 67), ("remainder", 440)]:
    print(f"  {c:28s} ~{v} MB predicted")
print()

g1 = named / best_tot >= 0.85
g2 = abs(accounted - g_res) / g_res <= 0.10
print(f"GATE named>=85%: {'PASS' if g1 else 'FAIL'}")
print(f"GATE accounted within 10% of reserved: {'PASS' if g2 else 'FAIL'}")
if g1 and g2:
    top3 = [c for c, _ in ranked[:3]]
    print(f"E0 VERDICT: MAP CONFIRMED — top-3 live-at-peak: {top3}")
    print("Levers green-lit in this order: " +
          ", ".join(f"E{i}" for i, c in
                     enumerate(["ffn", "mixer-proj", "norms"], 1)
                     if c in top3) or "re-examine")
else:
    print("E0 VERDICT: MAP INCOMPLETE — no lever gets built on this. "
          "Inspect 'other' and re-run.")


## 8. Save

In [ ]:
lines = []
lines.append(f"peak_alloc_MB={g_alloc:.0f}")
lines.append(f"peak_reserved_MB={g_res:.0f}")
lines.append(f"frag_gap_MB={g_frag:.0f}")
for tag, (cats, tot, nb) in _snaps.items():
    lines.append(f"--- live_at_{tag} total_MB={tot/1e6:.0f} blocks={nb} ---")
    for c, v in sorted(cats.items(), key=lambda kv: -kv[1]):
        lines.append(f"{tag} {c} MB={v/1e6:.1f}")
lines.append(f"ablation no-FFN delta_MB={p_full-p_noffn:.0f}")
lines.append(f"ablation no-mixer delta_MB={p_full-p_nomix:.0f}")
for bs, v in res.items():
    lines.append(f"batch_{bs} peak_MB={v:.0f}")
open("/content/v10a_results.txt", "w").write("\n".join(lines))
print(open("/content/v10a_results.txt").read())
from google.colab import files
files.download("/content/v10a_results.txt")
print("SAVE DONE")
